# Self-consistency (majority vote) on the baseline — submission generator

GRPO came in slightly under baseline (70.41% vs 71.60%), so we run self-consistency on the **untouched
base** `Qwen3-4B-Thinking-2507`. The pass-rate map showed exactly the structure SC exploits: many problems
the model gets right *most* of the time but not always — voting over n samples converts those to reliable.

How it works per problem:
- Sample **n** completions (temp 0.6, the Qwen3-recommended setting; diversity comes from sampling).
- Extract each sample's answer; **cluster by judger-equivalence** (so `\\boxed{5/8}` and `\\boxed{0.625}`
  vote together — the judger compares numerically at 1e-8). MC clusters by letter.
- Submit the full trace of a representative sample from the **largest** cluster, so the response carries a
  coherent reasoning trace whose final `\\boxed{}` is the majority answer.

Runs on `public.jsonl` (to measure lift on val vs the 71.60% baseline) and on `private.jsonl`
(to write the submission CSV). Pure inference; an A100-80GB is enough, an H100 is ~1.6× faster.

## 0. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/second_try/sft'      # data + harness/judger live here
OUT_DIR     = '/content/drive/MyDrive/second_try/self_consistency'
import os, sys
os.makedirs(OUT_DIR, exist_ok=True)
sys.path.insert(0, PROJECT_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('OUT_DIR    :', OUT_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/second_try/sft
OUT_DIR    : /content/drive/MyDrive/second_try/self_consistency


## 1. Install vLLM + grader deps (same stack as eval)

After this, **Runtime → Restart**, then run from section 2.

In [2]:
!pip install -q uv 2>&1 | tail -1
!uv pip install --system torch==2.7.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --system "vllm==0.9.2" "transformers==4.53.3" \
    sympy "antlr4-python3-runtime==4.11.1"
print("Install done. RESTART THE RUNTIME, then run from section 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 107.9 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 1.40s
Prepared 16 packages in 31.92s
Uninstalled 16 packages in 734ms
Installed 16 packages in 196ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.3.0.4
 - nvidia-cufile-cu12==1.13.1.3
 + nvidia-cufile-cu12==1.11.1.6
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.7.77
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.7.1.2
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.5.4.2
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0.6.3
 - nvidia-nccl-

## 2. Post-restart: version sanity + grading-path check

In [1]:
import os, sys
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
PROJECT_DIR = '/content/drive/MyDrive/second_try/sft'
OUT_DIR     = '/content/drive/MyDrive/second_try/self_consistency'
sys.path.insert(0, PROJECT_DIR)

import torch, transformers, vllm
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('vllm        :', vllm.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('GPU free    :', round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), 'GB')
assert transformers.__version__ == '4.53.3', 'wrong transformers — restart runtime'

from judger import Judger
_jt = Judger(strict_extract=False)
assert _jt.auto_judge(pred=r'\boxed{\frac{5}{8}}', gold=['5/8'], options=[[]]) is True, \
    'grading path broken'
# sanity: the judger treats 5/8 and 0.625 as equivalent (the basis of vote clustering)
assert _jt.auto_judge(pred=r'\boxed{0.625}', gold=['5/8'], options=[[]]) is True, \
    'numeric-equivalence path broken — clustering would over-split'
print('grading + numeric-equivalence path: OK')

torch       : 2.7.0+cu126
transformers: 4.53.3
vllm        : 0.9.2
device      : NVIDIA A100-SXM4-80GB
GPU free    : 84.65 GB
grading + numeric-equivalence path: OK


## 3. Config

`N_SAMPLES` is the only knob you normally touch. Use an **odd** value (5 or 7) so majority votes stay
decisive. On an A100-80GB, n=5 is comfortable; on an H100 (~1.6× faster) n=7 costs about the same wall
clock. Set `RUN_PRIVATE=False` first to measure val lift, then flip it to generate the submission.

`MODEL_TAG` namespaces the raw-sample archive. Every individual sample is saved under this tag so a
future run (e.g. a GRPO-improved or 0/4-distilled model) writes to its OWN archive instead of
overwriting. **Do not pool samples across tags blindly** — votes are draws from a specific policy;
mixing a weaker model's samples into a better model's vote degrades the better model. Cross-model
ensembling is a separate, deliberate choice (see §11), not automatic reuse.

In [2]:
MODEL_ID  = 'Qwen/Qwen3-4B-Thinking-2507'
MODEL_TAG = 'baseline'        # change per model: 'baseline', 'grpo_v1', 'distill04_v1', ...

PUBLIC_PATH  = f'{PROJECT_DIR}/public.jsonl'
PRIVATE_PATH = f'{PROJECT_DIR}/private.jsonl'      # <-- ensure private.jsonl is on Drive for submission
VAL_IDS_PATH = f'{PROJECT_DIR}/val_ids.json'

N_SAMPLES      = 7            # odd: 5 or 7
TEMPERATURE    = 0.6
TOP_P          = 0.95
TOP_K          = 20
MAX_GEN_TOKENS = 32768        # full thinking budget at inference
MAX_MODEL_LEN  = 40960
GPU_MEM_UTIL   = 0.90
SEED           = 151
CHUNK          = 40           # problems per generate() call; checkpoint after each (resumable)

RUN_PUBLIC   = False           # measure val lift vs 71.60% baseline
RUN_PRIVATE  = True          # flip to True to generate the submission CSV

# Raw per-sample archive (every individual sample, not just the voted winner), namespaced by model+n.
# Append-only and resumable; reusable for re-aggregating THIS model's samples (different n, tie rules,
# extractor) without regenerating. One file per split per model.
import os
RAW_DIR = f'{OUT_DIR}/raw_samples'
os.makedirs(RAW_DIR, exist_ok=True)
def raw_path(split):  return f'{RAW_DIR}/{MODEL_TAG}__{split}__samples.jsonl'

SUBMISSION_CSV = f'{OUT_DIR}/submission_{MODEL_TAG}_n{N_SAMPLES}.csv'
print(f'MODEL_TAG={MODEL_TAG} | N_SAMPLES={N_SAMPLES} (odd={N_SAMPLES % 2 == 1}) '
      f'temp={TEMPERATURE} cap={MAX_GEN_TOKENS}')
print(f'raw samples -> {RAW_DIR}/{MODEL_TAG}__<split>__samples.jsonl')
assert N_SAMPLES % 2 == 1, 'use an odd N_SAMPLES to avoid tie-heavy votes'

MODEL_TAG=baseline | N_SAMPLES=7 (odd=True) temp=0.6 cap=32768
raw samples -> /content/drive/MyDrive/second_try/self_consistency/raw_samples/baseline__<split>__samples.jsonl


## 4. Prompts (identical to Phase-1 baseline / eval)

In [3]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, "
    "then select the single best answer. After your reasoning, output ONLY the "
    "letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "The very last thing in your response must be that \\boxed{<letter>}."
)

def build_chat(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return [{"role": "system", "content": SYSTEM_PROMPT_MCQ},
                {"role": "user", "content": f"{question}\n\nOptions:\n{opts}"}]
    return [{"role": "system", "content": SYSTEM_PROMPT_MATH},
            {"role": "user", "content": question}]

## 5. Vote aggregation

Clusters the n samples by judger-equivalence and returns a representative response from the largest
cluster. MC uses the harness letter extractor; free-form uses `auto_judge(pred=response_i,
gold=extracted_j, options)` as the equivalence test — the exact relation the grader uses, so numerically
equal answers cluster and the submitted trace is one the grader will mark correct iff the cluster is right.

In [4]:
import re
import harness as H
from judger import Judger

_vote_judger = Judger(strict_extract=False)

def _mc_letter(text):
    return H.extract_letter(text)   # harness primary+fallback letter logic

def _ff_equiv(resp_a, gold_b):
    """True if response A grades correct against B's extracted answer (as gold)."""
    if not gold_b:
        return False
    try:
        return bool(_vote_judger.auto_judge(pred=resp_a, gold=gold_b,
                                            options=[[]] * len(gold_b)))
    except Exception:
        return False

def _ff_gold(text):
    """Extract a sample's answer as a gold-style list (split multi-answers)."""
    ext = _vote_judger.extract_ans(text)
    if not ext:
        return []
    parts = _vote_judger.split_by_comma(ext)
    return [p for p in parts if p != '']

def vote(samples, is_mc):
    """samples: list of {'text','truncated'}. Returns (winner_response, n_votes, n_clusters)."""
    if is_mc:
        clusters = {}                       # letter -> list of sample dicts
        for s in samples:
            L = _mc_letter(s['text'])
            clusters.setdefault(L, []).append(s)
        # drop the empty-letter bucket unless it's all we have
        keyed = {k: v for k, v in clusters.items() if k} or clusters
        best = max(keyed.values(), key=lambda v: (len(v),
                                                   sum(not x['truncated'] for x in v)))
        rep = next((x for x in best if not x['truncated']), best[0])
        return rep['text'], len(best), len(keyed)

    # free-form: cluster by judger-equivalence
    clusters = []                           # each: {'gold':[...], 'members':[...]}
    for s in samples:
        g = _ff_gold(s['text'])
        placed = False
        for c in clusters:
            if _ff_equiv(s['text'], c['gold']) or (g and _ff_equiv(c['rep']['text'], g)):
                c['members'].append(s)
                placed = True
                break
        if not placed:
            clusters.append({'gold': g, 'rep': s, 'members': [s]})
    if not clusters:
        return samples[0]['text'], 0, 0
    best = max(clusters, key=lambda c: (len(c['members']),
                                        sum(not x['truncated'] for x in c['members'])))
    rep = next((x for x in best['members'] if not x['truncated']), best['members'][0])
    return rep['text'], len(best['members']), len(clusters)

# quick self-test of the equivalence-clustering on a known pair
_a = r"reasoning... \boxed{\frac{5}{8}}"
_b = r"different reasoning... \boxed{0.625}"
_c = r"wrong... \boxed{\frac{1}{2}}"
_w, _v, _n = vote([{'text': _a, 'truncated': False},
                   {'text': _b, 'truncated': False},
                   {'text': _c, 'truncated': False}], is_mc=False)
assert _v == 2 and _n == 2, f'clustering self-test failed: votes={_v} clusters={_n}'
print('vote clustering self-test: OK (5/8 and 0.625 clustered; 1/2 separate)')

vote clustering self-test: OK (5/8 and 0.625 clustered; 1/2 separate)


## 6. Load model + sampler

In [5]:
import json
import harness as H
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
llm = LLM(model=MODEL_ID, dtype='bfloat16', trust_remote_code=True,
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
          seed=SEED, enforce_eager=True)
sp = SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, top_p=TOP_P,
                    top_k=TOP_K, min_p=0.0, max_tokens=MAX_GEN_TOKENS, seed=SEED)
print('model loaded')

INFO 05-31 01:24:38 [__init__.py:244] Automatically detected platform cuda.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-31 01:25:05 [config.py:841] This model supports multiple tasks: {'reward', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 05-31 01:25:05 [config.py:1472] Using max model len 40960
INFO 05-31 01:25:05 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-31 01:25:05 [cuda.py:102] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model loaded


## 7. Core run loop (archives every raw sample, resumable)

For each problem we save **all n individual samples** (full text + finish reason) to the per-model raw
archive, then vote. Re-running resumes from the archive (completed ids skipped). The raw file is the
durable asset: you can re-vote THIS model's samples later with a different n / tie rule / extractor
without regenerating. It does NOT auto-mix with other models' archives.

In [6]:
import json, os

def run_split(rows, raw_out, label):
    # Resume from the raw archive: a problem is 'done' once its samples are written.
    done = set()
    if os.path.exists(raw_out):
        for line in open(raw_out):
            line = line.strip()
            if line:
                done.add(json.loads(line)['id'])
    todo = [r for r in rows if r['id'] not in done]
    print(f'[{label}] total={len(rows)} archived={len(done)} todo={len(todo)}')

    with open(raw_out, 'a') as fout:
        for ci in range(0, len(todo), CHUNK):
            chunk = todo[ci:ci + CHUNK]
            prompts = [tok.apply_chat_template(build_chat(r['question'], r.get('options')),
                                               tokenize=False, add_generation_prompt=True)
                       for r in chunk]
            outs = llm.generate(prompts, sp)
            for r, out in zip(chunk, outs):
                rec = {
                    'id': r['id'],
                    'model_tag': MODEL_TAG,
                    'is_mc': bool(r.get('options')),
                    'n_samples': len(out.outputs),
                    'temperature': TEMPERATURE,
                    'max_gen_tokens': MAX_GEN_TOKENS,
                    # every individual sample, full text — this is the reusable asset
                    'samples': [{'text': o.text,
                                 'truncated': (o.finish_reason == 'length'),
                                 'n_tok': len(o.token_ids)} for o in out.outputs],
                }
                fout.write(json.dumps(rec) + '\n')
            fout.flush(); os.fsync(fout.fileno())
            print(f'  [{label}] {min(ci+CHUNK, len(todo))}/{len(todo)} archived')
    print(f'[{label}] raw samples -> {raw_out}')


def aggregate(raw_out):
    # Re-vote from an archived raw-sample file. Pure; safe to call repeatedly with a
    # different N_SAMPLES (uses up to the first N_SAMPLES archived samples per problem).
    voted = {}
    for line in open(raw_out):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        samples = rec['samples'][:N_SAMPLES]          # re-aggregation can use fewer than archived
        winner, n_votes, n_clusters = vote(samples, rec['is_mc'])
        voted[rec['id']] = {
            'response': winner, 'n_votes': n_votes, 'n_clusters': n_clusters,
            'is_mc': rec['is_mc'],
            'n_truncated': sum(s['truncated'] for s in samples),
        }
    return voted

## 8. Public run → measure val lift vs 71.60% baseline

In [7]:
if RUN_PUBLIC:
    public = H.load_jsonl(PUBLIC_PATH)
    run_split(public, raw_path('public'), 'public')
    voted = aggregate(raw_path('public'))

    # Score the voted predictions on the val split only (apples-to-apples vs baseline).
    val_ids = set(json.load(open(VAL_IDS_PATH)))
    val_rows = [r for r in public if r['id'] in val_ids]
    preds = [{'id': r['id'], 'response': voted[r['id']]['response']} for r in val_rows]
    trunc_ids = {r['id'] for r in val_rows
                 if voted[r['id']]['n_truncated'] == N_SAMPLES}   # all samples truncated
    summary, _ = H.score(preds, val_rows, truncated_ids=trunc_ids)

    print(f'\n### SELF-CONSISTENCY [{MODEL_TAG}] n={N_SAMPLES} — VAL accuracy ###')
    H.print_summary(summary)
    BASELINE = {'mc': 0.8036, 'free_single': 0.6667, 'free_multi': 0.6774, 'overall': 0.7160}
    print('\nLift vs Phase-1 baseline:')
    for b in ['mc', 'free_single', 'free_multi', 'overall']:
        new = summary[b]['acc']; d = (new - BASELINE[b]) * 100
        sign = '+' if d >= 0 else ''
        print(f'  {b:<12} {new*100:6.2f}%  ({sign}{d:.2f} pp)')
else:
    print('RUN_PUBLIC is False — skipping val measurement')

RUN_PUBLIC is False — skipping val measurement


## 9. Private run → submission CSV

Flip `RUN_PRIVATE=True` in §3 once the val lift looks good. Writes the competition CSV with proper
quoting (responses contain commas/newlines). The `response` is the full voted trace; the grader extracts
the final `\\boxed{}` from it.

In [8]:
import csv, json, os
if RUN_PRIVATE:
    assert os.path.exists(PRIVATE_PATH), f'private.jsonl not found at {PRIVATE_PATH}'
    private = H.load_jsonl(PRIVATE_PATH)
    run_split(private, raw_path('private'), 'private')
    voted = aggregate(raw_path('private'))

    missing = [r['id'] for r in private if r['id'] not in voted]
    assert not missing, f'{len(missing)} private ids missing from voted output, e.g. {missing[:5]}'

    with open(SUBMISSION_CSV, 'w', newline='') as f:
        w = csv.writer(f, quoting=csv.QUOTE_ALL)
        w.writerow(['id', 'response'])
        for r in private:
            w.writerow([r['id'], voted[r['id']]['response']])
    print(f'wrote submission: {SUBMISSION_CSV}  ({len(private)} rows)')
else:
    print('RUN_PRIVATE is False — skipping submission generation')

[private] total=943 archived=0 todo=943


Adding requests:   0%|          | 0/40 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [private] 40/943 archived


Adding requests:   0%|          | 0/40 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/280 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [private] 920/943 archived


Adding requests:   0%|          | 0/23 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/161 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  [private] 943/943 archived
[private] raw samples -> /content/drive/MyDrive/second_try/self_consistency/raw_samples/baseline__private__samples.jsonl
wrote submission: /content/drive/MyDrive/second_try/self_consistency/submission_baseline_n7.csv  (943 rows)


## 10. Notes

- **Raw archive** (`raw_samples/{MODEL_TAG}__{split}__samples.jsonl`): every individual sample, full
  text + finish reason + token count. This is the reusable asset. To re-aggregate THIS model later with
  a different `N_SAMPLES`, tie rule, or extractor, just rerun §8/§9's `aggregate()` on the archive — no
  regeneration. Lowering `N_SAMPLES` re-votes a subset for free; raising it beyond what's archived means
  generating the extra samples (the run loop appends).
- `n_votes` / `n_clusters` per problem are diagnostics: mean `n_votes` near 1 means the model rarely
  agrees with itself (little SC benefit); near `N_SAMPLES` means strong consensus.
- Ties break toward the larger cluster, then toward a non-truncated representative. Odd `N_SAMPLES`
  keeps ties rare.

## 11. Reusing votes with a *better* model later (do this deliberately, not by accident)

A future better model (GRPO-improved, 0/4-distilled) should run with its **own** `MODEL_TAG`, writing a
**separate** archive. Do **not** pool its samples with the baseline's by majority vote — votes are draws
from a specific policy, and stale baseline samples (more often wrong) would drag the better model's
consensus down. Naive pooling makes a better model score *worse*.

If you want to combine models, that's *ensembling*, and it's a deliberate design choice — e.g.
**weight each model's votes by its measured val accuracy**, or take the better model's answer and only
fall back to the baseline's consensus when the better model has no majority. Sketch:

```python
# weighted cross-model vote (illustrative — validate on val before trusting it)
W = {'baseline': 0.716, 'grpo_v1': 0.74}     # per-model val accuracy as weight
per_model = {tag: aggregate_raw(raw_path_for(tag, split)) for tag in W}
# for each id: tally cluster weights = sum of W[tag] over that model's winning votes,
# then pick the answer with the highest weighted tally.
```

The point: keep archives **per model and separate**, then choose how (or whether) to combine them, and
**measure any combination on val** before it becomes a submission. Automatic pooling is the trap; an
accuracy-weighted ensemble you've validated is a legitimate lift.